# Semana 8 — Projeto Prático de Data Warehouse e Modelagem Dimensional

Mini Data Warehouse de e-commerce usando o dataset Olist.

## Objetivos

- Diferenciar OLTP e OLAP
- Executar ETL com Pandas
- Criar dimensões e tabela fato
- Identificar PK, FK e Surrogate Key
- Criar Star Schema
- Consultar o modelo com SQL
- Interpretar métricas de BI

## Arquivos necessários

Baixe no Kaggle o dataset Brazilian E-Commerce Public Dataset by Olist e coloque na pasta do notebook:

- olist_orders_dataset.csv
- olist_order_items_dataset.csv
- olist_customers_dataset.csv
- olist_products_dataset.csv
- product_category_name_translation.csv

In [2]:
import pandas as pd
import duckdb
import matplotlib.pyplot as plt

pd.set_option('display.max_columns', 100)

# 1. Cenário de negócio

O diretor quer responder:

1. Qual foi o faturamento total?
2. Quais estados mais faturaram?
3. Quais categorias mais venderam?
4. Qual foi o ticket médio?
5. Como o faturamento evoluiu ao longo do tempo?

# 2. Extract — carregando fontes OLTP

In [4]:
pedidos = pd.read_csv('/Users/Dell/Documents/GitHub/Semana_9/Dataset/olist_orders_dataset.csv')
itens = pd.read_csv('/Users/Dell/Documents/GitHub/Semana_9/Dataset/olist_order_items_dataset.csv')
clientes = pd.read_csv('/Users/Dell/Documents/GitHub/Semana_9/Dataset/olist_customers_dataset.csv')
produtos = pd.read_csv('/Users/Dell/Documents/GitHub/Semana_9/Dataset/olist_products_dataset.csv')
traducao_categoria = pd.read_csv('/Users/Dell/Documents/GitHub/Semana_9/Dataset/product_category_name_translation.csv')

print('orders:', pedidos.shape)
print('items:', itens.shape)
print('customers:', clientes.shape)
print('products:', produtos.shape)
print('category_translation:', traducao_categoria.shape)

orders: (99441, 8)
items: (112650, 7)
customers: (99441, 5)
products: (32951, 9)
category_translation: (71, 2)


## Exercício 1 🟢 Fácil — OLTP

Responda:

1. Por que essas tabelas podem ser consideradas fontes OLTP?
2. Qual tabela registra o pedido?
3. Qual tabela registra os itens de cada pedido?
4. Qual tabela descreve os clientes?
5. Qual tabela descreve os produtos?

# 3. Análise exploratória inicial

In [5]:
pedidos.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [14]:
itens.head()

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14


In [15]:
clientes.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [16]:
produtos.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.0,287.0,1.0,225.0,16.0,10.0,14.0
1,3aa071139cb16b67ca9e5dea641aaa2f,artes,44.0,276.0,1.0,1000.0,30.0,18.0,20.0
2,96bd76ec8810374ed1b65e291975717f,esporte_lazer,46.0,250.0,1.0,154.0,18.0,9.0,15.0
3,cef67bcfe19066a932b7673e239eb23d,bebes,27.0,261.0,1.0,371.0,26.0,4.0,26.0
4,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,37.0,402.0,4.0,625.0,20.0,17.0,13.0


## Exercício 2 🟢 Fácil — Explorando os dados

Use `.info()`, `.shape` e `.isnull().sum()` para investigar as tabelas.

In [ ]:
# Investigue a tabela orders
# SUA RESPOSTA AQUI

In [ ]:
# Investigue a tabela items
# SUA RESPOSTA AQUI

In [ ]:
# Investigue a tabela customers
# SUA RESPOSTA AQUI

In [ ]:
# Investigue a tabela products
# SUA RESPOSTA AQUI

# 4. Transform — tratamento dos dados

Transformações comuns: converter tipos, remover duplicidades, tratar nulos, padronizar nomes e criar colunas derivadas.

In [17]:
pedidos['order_purchase_timestamp'] = pd.to_datetime(pedidos['order_purchase_timestamp'])
pedidos['order_approved_at'] = pd.to_datetime(pedidos['order_approved_at'])
pedidos['order_delivered_customer_date'] = pd.to_datetime(pedidos['order_delivered_customer_date'])
pedidos['order_estimated_delivery_date'] = pd.to_datetime(pedidos['order_estimated_delivery_date'])

pedidos[['order_purchase_timestamp', 'order_approved_at', 'order_delivered_customer_date']].head()

,order_purchase_timestamp,order_approved_at,order_delivered_customer_date
0,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-10 21:25:13
1,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-08-07 15:27:45
2,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-17 18:06:29
3,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-12-02 00:28:42
4,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-16 18:17:02


## Exercício 3 🟢 Fácil — Criando atributos de tempo

Crie as colunas `ano`, `mes`, `dia`, `trimestre` e `ano_mes` a partir de `order_purchase_timestamp`.

In [ ]:
# SUA RESPOSTA AQUI



## Exercício 4 🟢 Fácil — Remoção de duplicidades

Verifique se existem duplicidades nas tabelas principais e remova-as.

In [ ]:
# SUA RESPOSTA AQUI



## Exercício 5 🟡 Médio — Filtro de pedidos válidos

Considere apenas pedidos com status `delivered` e crie `produtos_entregue`.

In [ ]:
# SUA RESPOSTA AQUI



# 5. Modelagem dimensional

Modelo esperado:

```text
dim_cliente
     |
dim_produto -- fato_vendas -- dim_tempo
```

A tabela fato guarda eventos mensuráveis. As dimensões guardam contexto.

# 6. Criando a dimensão cliente

In [18]:
dim_cliente = clientes[['customer_id', 'customer_unique_id', 'customer_city', 'customer_state']].drop_duplicates().copy()
dim_cliente = dim_cliente.reset_index(drop=True)
dim_cliente['cliente_sk'] = dim_cliente.index + 1
dim_cliente = dim_cliente[['cliente_sk', 'customer_id', 'customer_unique_id', 'customer_city', 'customer_state']]
dim_cliente.head()

,cliente_sk,customer_id,customer_unique_id,customer_city,customer_state
0,1,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,franca,SP
1,2,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,sao bernardo do campo,SP
2,3,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,sao paulo,SP
3,4,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,mogi das cruzes,SP
4,5,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,campinas,SP


## Exercício 6 🟡 Médio — PK, SK e chave natural

1. Qual coluna é a Surrogate Key da dimensão cliente?
2. Qual coluna veio do sistema de origem?
3. Por que não usar CPF, e-mail ou código da origem como PK física do DW?

# 7. Criando a dimensão produto

In [19]:
dim_produto = produtos[['product_id', 'product_category_name']].drop_duplicates().copy()
dim_produto = dim_produto.merge(traducao_categoria, on='product_category_name', how='left')
dim_produto['product_category_name_english'] = dim_produto['product_category_name_english'].fillna('unknown')
dim_produto = dim_produto.reset_index(drop=True)
dim_produto['produto_sk'] = dim_produto.index + 1
dim_produto = dim_produto[['produto_sk', 'product_id', 'product_category_name', 'product_category_name_english']]
dim_produto.head()

,produto_sk,product_id,product_category_name,product_category_name_english
0,1,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,perfumery
1,2,3aa071139cb16b67ca9e5dea641aaa2f,artes,art
2,3,96bd76ec8810374ed1b65e291975717f,esporte_lazer,sports_leisure
3,4,cef67bcfe19066a932b7673e239eb23d,bebes,baby
4,5,9dc1a7de274444849c219cff195d0b71,utilidades_domesticas,housewares


## Exercício 7 🟡 Médio — Star Schema vs Snowflake

1. Esse modelo se parece mais com Star Schema ou Snowflake?
2. Como ficaria se categoria fosse uma tabela separada?
3. Qual opção costuma ser mais simples para Power BI?

# 8. Criando a dimensão tempo

In [20]:
#verifica se tem duplicidade de data e copia a tabela para criar a dimensão tempo

dim_tempo = produtos_entregues[['order_purchase_timestamp']].drop_duplicates().copy()

### Crie as colunas necessárias para a dimensão tempo
dim_tempo['data'] = dim_tempo['order_purchase_timestamp'].dt.date #cria a coluna data apenas com a data, sem o horário
dim_tempo['ano'] = dim_tempo['order_purchase_timestamp'].dt.year #cria a coluna ano apenas com o ano da data
dim_tempo['mes'] = dim_tempo['order_purchase_timestamp'].dt.month #cria a coluna mes apenas com o mês da data
dim_tempo['dia'] = dim_tempo['order_purchase_timestamp'].dt.day #cria a coluna dia apenas com o dia da data
dim_tempo['trimestre'] = dim_tempo['order_purchase_timestamp'].dt.quarter #cria a coluna trimestre apenas com o trimestre da data


dim_tempo['ano_mes'] = dim_tempo['order_purchase_timestamp'].dt.to_period('M').astype(str) #cria a coluna ano_mes apenas com o ano e o mês da data, no formato "YYYY-MM"
dim_tempo = dim_tempo.drop_duplicates('data').reset_index(drop=True) #remove as linhas duplicadas com base na coluna data e reseta o índice
dim_tempo['tempo_sk'] = dim_tempo.index + 1 #cria a coluna tempo_sk com os valores sequenciais
dim_tempo = dim_tempo[['tempo_sk', 'data', 'ano', 'mes', 'dia', 'trimestre', 'ano_mes']] #reordena as colunas da dimensão tempo
dim_tempo.head() #exibe as primeiras linhas da dimensão tempo

NameError: name 'produtos_entregues' is not defined

## Exercício 8 🟡 Médio — Dimensão tempo

1. Por que uma dimensão tempo é útil em BI?
2. Que análises ela permite?
3. Qual coluna é a SK da dimensão tempo?

# 9. Criando a tabela fato

Evento: um item vendido dentro de um pedido.

Granularidade: uma linha por item vendido em um pedido.

In [ ]:
fato_vendas = items[['order_id', 'order_item_id', 'product_id', 'price', 'freight_value']].copy()
fato_vendas = fato_vendas.merge(pedidos_entregue[['order_id', 'customer_id', 'order_purchase_timestamp']], on='order_id', how='inner')
fato_vendas.head()

## Exercício 9 🟡 Médio — Relacionando fato e dimensões

Complete a construção da `fato_vendas` trazendo `cliente_sk`, `produto_sk` e `tempo_sk`.

In [ ]:
# SUA RESPOSTA AQUI

## Exercício 10 🟡 Médio — Métricas da fato

Crie `valor_produto`, `valor_frete` e `valor_total`.

In [ ]:
# SUA RESPOSTA AQUI

## Exercício 11 🔴 Difícil — Fato final

Deixe a fato com: `order_id`, `order_item_id`, `cliente_sk`, `produto_sk`, `tempo_sk`, `valor_produto`, `valor_frete`, `valor_total`.

Depois, crie `venda_sk`.

In [ ]:
# SUA RESPOSTA AQUI

# 10. Validando o Star Schema

## Exercício 12 🔴 Difícil — Documentação do modelo

| Tabela | Tipo | PK | FK | Descrição |
|---|---|---|---|---|
| dim_cliente |  |  |  |  |
| dim_produto |  |  |  |  |
| dim_tempo |  |  |  |  |
| fato_vendas |  |  |  |  |

# 11. SQL Analítico com DuckDB

In [ ]:
con = duckdb.connect()
con.register('dim_cliente', dim_cliente) # registra a dimensão cliente no banco de dados
con.register('dim_produto', dim_produto) # registra a dimensão produto no banco de dados
con.register('dim_tempo', dim_tempo) # registra a dimensão tempo no banco de dados
con.register('fato_vendas', fato_vendas_final) # registra a tabela fato vendas no banco de dados

## Consulta exemplo — Faturamento total

In [ ]:
con.sql("""
SELECT SUM(valor_total) AS faturamento_total
FROM fato_vendas
""").df() # consulta para calcular o faturamento total da loja, somando a coluna valor_total da tabela fato_vendas

## Exercício 13 🟡 Médio — Faturamento por estado

In [ ]:
con.sql("""
-- SUA CONSULTA SQL AQUI
""").df() 

## Exercício 14 🟡 Médio — Top 10 categorias

In [ ]:
con.sql("""
-- SUA CONSULTA SQL AQUI
""").df() 

## Exercício 15 🔴 Difícil — Ticket médio por estado

In [ ]:
con.sql("""
-- SUA CONSULTA SQL AQUI
""").df()

## Exercício 16 🔴 Difícil — Evolução mensal de faturamento

In [ ]:
con.sql("""
-- SUA CONSULTA SQL AQUI
""").df()

# 12. Visualização simples

In [ ]:
faturamento_mensal = con.sql("""
SELECT t.ano_mes, SUM(f.valor_total) AS faturamento_total
FROM fato_vendas f
JOIN dim_tempo t ON f.tempo_sk = t.tempo_sk
GROUP BY t.ano_mes
ORDER BY t.ano_mes
""").df()
faturamento_mensal.head()

In [ ]:
plt.figure(figsize=(12,5))
plt.plot(faturamento_mensal['ano_mes'], faturamento_mensal['faturamento_total'])
plt.xticks(rotation=90)
plt.title('Evolução Mensal do Faturamento')
plt.xlabel('Ano-Mês')
plt.ylabel('Faturamento')
plt.show()

## Exercício 17 🔴 Difícil — Visualização

Crie um gráfico de barras com o faturamento por estado.

In [ ]:
# SUA RESPOSTA AQUI

# 13. Snowflake Schema — desafio conceitual

No modelo atual, `dim_produto` possui categoria dentro da própria dimensão.

Em Snowflake, poderíamos separar `dim_produto` e `dim_categoria`.

## Exercício 18 🔴 Difícil

1. Qual vantagem do Snowflake nesse caso?
2. Qual desvantagem?
3. Para Power BI, você escolheria Star ou Snowflake? Justifique.

# 14. Desafio Final — Caso Executivo

O diretor pediu uma análise com:

1. Faturamento total
2. Top 5 estados por faturamento
3. Top 10 categorias
4. Ticket médio geral
5. Evolução mensal do faturamento
6. Explicação do modelo dimensional usado
7. Justificativa de por que a solução é OLAP

## Entrega

- Notebook preenchido
- Print ou desenho do Star Schema
- Resposta executiva com 5 a 10 linhas